# KNN Model Configuration Benchmark

## Objective

Determine the optimal configuration for the KNN-based missing attribute estimation model.

## Experiment 1: Encoding Strategy

- OneHotEncoder
- OrdinalEncoder

Evaluation:
- Memory usage
- Feature dimensionality
- Execution time

Conclusion:
OneHotEncoder selected.

---

## Experiment 2: Neighbor Search Algorithm

Algorithms evaluated:
- auto
- brute
- kd_tree
- ball_tree

Evaluation:
- Training time
- Prediction time

Conclusion:
<Brute>

In [1]:
import sqlite3
import time

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.neighbors import NearestNeighbors

In [2]:
# Load the engineered dataset
conn = sqlite3.connect("pakwheels_dataset.db")

df = pd.read_sql_query(
    "SELECT * FROM market_analysis_cars",
    conn
)

conn.close()


# Keep only the features used by the KNN estimator
required_columns = [
    "Company",
    "Model",
    "City",
    "Year",
    "Mileage_KM",
    "Engine_CC",
    "Fuel_Type",
    "Transmission",
    "Price_PKR"
]

df = df[required_columns].dropna()


# Convert numeric columns to integer
df["Year"] = df["Year"].astype(int)
df["Mileage_KM"] = df["Mileage_KM"].astype(int)
df["Engine_CC"] = df["Engine_CC"].astype(int)
df["Price_PKR"] = df["Price_PKR"].astype(int)


# Define categorical and numeric feature groups
categorical_columns = [
    "Company",
    "Model",
    "City",
    "Fuel_Type",
    "Transmission"
]

numeric_columns = [
    "Year",
    "Mileage_KM",
    "Engine_CC",
    "Price_PKR"
]


print(f"Dataset Shape : {df.shape}")
print(df.head())


Dataset Shape : (2526, 9)
    Company    Model    City  Year  Mileage_KM  Engine_CC Fuel_Type  \
0    Toyota  Corolla  Lahore  2018       92000       1300    Petrol   
1  Daihatsu     Mira  Lahore  2022       16000        660    Petrol   
2    Toyota  Corolla  Lahore  2019      109000       1300    Petrol   
3    Toyota    Yaris  Lahore  2021      117855       1500    Petrol   
4     Honda     City  Lahore  2018      156181       1300    Petrol   

  Transmission  Price_PKR  
0    Automatic    3990000  
1    Automatic    3890000  
2       Manual    3750000  
3    Automatic    4450000  
4    Automatic    3500000  


In [3]:
# ==========================================================
# Benchmark Configuration
# ==========================================================

# Number of benchmark repetitions
NUM_RUNS = 30

# Same K value used in the KNN estimator
K_NEIGHBORS = 10

# Use every vehicle in the dataset as a query point.
# This removes the need to choose a specific row and
# avoids sampling bias.
query_dataset = df.copy()

print(f"Benchmark repetitions : {NUM_RUNS}")
print(f"K Neighbors           : {K_NEIGHBORS}")
print(f"Query Vehicles        : {len(query_dataset)}")

Benchmark repetitions : 30
K Neighbors           : 10
Query Vehicles        : 2526


In [4]:
def run_benchmark(encoder_name, encoder):
    """
    Benchmark a categorical encoder using the same KNN workflow
    as the project implementation.
    """

    preprocessing_times = []
    fit_times = []
    prediction_times = []
    total_times = []

    feature_dimensions = []
    memory_usage = []

    # Use the same query every run
    query_row = df.iloc[[0]]
    
    # -----------------------------
    # Build preprocessing pipeline
    # -----------------------------
    preprocessor = ColumnTransformer(
        transformers=[
        ("cat", encoder, categorical_columns),
        ("num", StandardScaler(), numeric_columns)
       ],
        remainder="drop"
        )

    for run in range(NUM_RUNS + 1):
        preprocessor = ColumnTransformer(
            transformers=[
            ("cat", encoder, categorical_columns),
            ("num", StandardScaler(), numeric_columns)
            ],
            remainder="drop"
            )
        
        
        if run == 0:
        # Warm-up iteration
            continue


        total_start = time.perf_counter()

        # -----------------------------
        # Preprocessing
        # -----------------------------
        preprocessing_start = time.perf_counter()

        X_dataset = preprocessor.fit_transform(df)

        X_query = preprocessor.transform(query_row)

        preprocessing_end = time.perf_counter()

        # -----------------------------
        # Feature information
        # -----------------------------
        feature_dimensions.append(X_dataset.shape[1])

        if hasattr(X_dataset, "nbytes"):
            memory_usage.append(
                X_dataset.nbytes / (1024 ** 2)
            )
        else:
            memory_usage.append(
                X_dataset.data.nbytes / (1024 ** 2)
            )

        # -----------------------------
        # Build KNN
        # -----------------------------
        knn = NearestNeighbors(
            n_neighbors=K_NEIGHBORS,
            metric="euclidean"
        )

        fit_start = time.perf_counter()

        knn.fit(X_dataset)

        fit_end = time.perf_counter()

        # -----------------------------
        # Query
        # -----------------------------
        prediction_start = time.perf_counter()

        knn.kneighbors(X_query)

        prediction_end = time.perf_counter()

        total_end = time.perf_counter()

        preprocessing_times.append(
            preprocessing_end - preprocessing_start
        )

        fit_times.append(
            fit_end - fit_start
        )

        prediction_times.append(
            prediction_end - prediction_start
        )

        total_times.append(
            total_end - total_start
        )
    return {
        "Encoder": encoder_name,

        "Feature Dimensions": feature_dimensions,
        "Memory (MB)": memory_usage,

        "Preprocessing Time": preprocessing_times,
        "Fit Time": fit_times,
        "Prediction Time": prediction_times,
        "Total Time": total_times
    }
    

In [5]:
onehot_results = run_benchmark(
    encoder_name="OneHotEncoder",
    encoder=OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
)

print("OneHotEncoder Benchmark Completed\n")

for key, value in onehot_results.items():
    if isinstance(value, list):
        print(f"{key}:")
        print(value)
        print()
    else:
        print(f"{key}: {value}")

OneHotEncoder Benchmark Completed

Encoder: OneHotEncoder
Feature Dimensions:
[206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206, 206]

Memory (MB):
[3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125, 3.970001220703125]

Preprocessing Time:
[0.0256874999977299, 0.025380700004461687, 0.026817700003448408, 0.026404800002637785, 0.023992299997189548, 0.02253950000158511, 0.026644600002327934, 0.01522290000

In [6]:
ordinal_results = run_benchmark(
    encoder_name="OrdinalEncoder",
    encoder=OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
)

print("OrdinalEncoder Benchmark Completed")

OrdinalEncoder Benchmark Completed


In [7]:
def summarize_results(results):
    return {
        "Encoder": results["Encoder"],

        "Feature Dimensions": int(np.mean(results["Feature Dimensions"])),
        "Memory (MB)": round(np.mean(results["Memory (MB)"]), 6),

        "Preprocessing Mean (s)": round(np.mean(results["Preprocessing Time"]), 6),
        "Preprocessing Std (s)": round(np.std(results["Preprocessing Time"]), 6),
        "Preprocessing Min (s)": round(np.min(results["Preprocessing Time"]), 6),
        "Preprocessing Max (s)": round(np.max(results["Preprocessing Time"]), 6),

        "Fit Mean (s)": round(np.mean(results["Fit Time"]), 6),
        "Fit Std (s)": round(np.std(results["Fit Time"]), 6),
        "Fit Min (s)": round(np.min(results["Fit Time"]), 6),
        "Fit Max (s)": round(np.max(results["Fit Time"]), 6),

        "Prediction Mean (s)": round(np.mean(results["Prediction Time"]), 6),
        "Prediction Std (s)": round(np.std(results["Prediction Time"]), 6),
        "Prediction Min (s)": round(np.min(results["Prediction Time"]), 6),
        "Prediction Max (s)": round(np.max(results["Prediction Time"]), 6),

        "Total Mean (s)": round(np.mean(results["Total Time"]), 6),
        "Total Std (s)": round(np.std(results["Total Time"]), 6),
        "Total Min (s)": round(np.min(results["Total Time"]), 6),
        "Total Max (s)": round(np.max(results["Total Time"]), 6)
    }


summary_df = pd.DataFrame([
    summarize_results(onehot_results),
    summarize_results(ordinal_results)
])

print("\n========== Encoding Benchmark Summary ==========\n")

display(summary_df)


========== Encoding Benchmark Summary ==========



,Encoder,Feature Dimensions,Memory (MB),Preprocessing Mean (s),Preprocessing Std (s),Preprocessing Min (s),Preprocessing Max (s),Fit Mean (s),Fit Std (s),Fit Min (s),Fit Max (s),Prediction Mean (s),Prediction Std (s),Prediction Min (s),Prediction Max (s),Total Mean (s),Total Std (s),Total Min (s),Total Max (s)
0,OneHotEncoder,206,3.970001,0.022664,0.00452,0.014516,0.027829,0.001156,0.000282,0.000706,0.001799,0.093228,0.496619,0.000676,2.767602,0.117377,0.497078,0.016867,2.794108
1,OrdinalEncoder,9,0.173447,0.012858,0.00182,0.011075,0.018488,0.002987,0.000439,0.002626,0.004937,0.000630,0.000252,0.000403,0.001470,0.016492,0.002276,0.014510,0.024364


In [8]:
#######################################
# Benchmark different KNN search algorithms
#######################################

def run_algorithm_benchmark(algorithm_name, knn_algorithm):
    """
    Benchmark different KNN search algorithms
    while keeping preprocessing identical.
    """

    preprocessing_times = []
    fit_times = []
    prediction_times = []
    total_times = []

    feature_dimensions = []
    memory_usage = []

    # Keep the encoder fixed
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

    # Same query every run
    query_row = df.iloc[[0]]

    for run in range(NUM_RUNS + 1):

        # Fresh preprocessing pipeline every run
        preprocessor = ColumnTransformer(
            transformers=[
                ("cat", encoder, categorical_columns),
                ("num", StandardScaler(), numeric_columns)
            ],
            remainder="drop"
        )

        # -----------------------------
        # Warm-up iteration
        # -----------------------------
        if run == 0:

            X_dataset = preprocessor.fit_transform(df)
            X_query = preprocessor.transform(query_row)

            knn = NearestNeighbors(
                n_neighbors=K_NEIGHBORS,
                metric="euclidean",
                algorithm=knn_algorithm
            )

            knn.fit(X_dataset)

            print(f"Requested: {knn_algorithm} | Actual: {knn._fit_method}")

            knn.kneighbors(X_query)

            continue

        total_start = time.perf_counter()

        # -----------------------------
        # Preprocessing
        # -----------------------------
        preprocessing_start = time.perf_counter()

        X_dataset = preprocessor.fit_transform(df)
        X_query = preprocessor.transform(query_row)

        preprocessing_end = time.perf_counter()

        # -----------------------------
        # Feature information
        # -----------------------------
        feature_dimensions.append(X_dataset.shape[1])

        if hasattr(X_dataset, "nbytes"):
            memory_usage.append(
                X_dataset.nbytes / (1024 ** 2)
            )
        else:
            memory_usage.append(
                X_dataset.data.nbytes / (1024 ** 2)
            )

        # -----------------------------
        # Build KNN
        # -----------------------------
        knn = NearestNeighbors(
            n_neighbors=K_NEIGHBORS,
            metric="euclidean",
            algorithm=knn_algorithm
        )

        fit_start = time.perf_counter()

        knn.fit(X_dataset)

        fit_end = time.perf_counter()

        # -----------------------------
        # Query
        # -----------------------------
        prediction_start = time.perf_counter()

        knn.kneighbors(X_query)

        prediction_end = time.perf_counter()

        total_end = time.perf_counter()

        preprocessing_times.append(
            preprocessing_end - preprocessing_start
        )

        fit_times.append(
            fit_end - fit_start
        )

        prediction_times.append(
            prediction_end - prediction_start
        )

        total_times.append(
            total_end - total_start
        )

    return {

        "Algorithm": algorithm_name,

        "Feature Dimensions": feature_dimensions,
        "Memory (MB)": memory_usage,

        "Preprocessing Time": preprocessing_times,
        "Fit Time": fit_times,
        "Prediction Time": prediction_times,
        "Total Time": total_times
    }

In [9]:
#######################################
# Run KNN Algorithm Benchmarks
#######################################

auto_results = run_algorithm_benchmark(
    algorithm_name="auto",
    knn_algorithm="auto"
)

print("Auto Benchmark Completed")


brute_results = run_algorithm_benchmark(
    algorithm_name="brute",
    knn_algorithm="brute"
)

print("Brute Benchmark Completed")


kd_tree_results = run_algorithm_benchmark(
    algorithm_name="kd_tree",
    knn_algorithm="kd_tree"
)

print("KD-Tree Benchmark Completed")


ball_tree_results = run_algorithm_benchmark(
    algorithm_name="ball_tree",
    knn_algorithm="ball_tree"
)

print("Ball-Tree Benchmark Completed")

Requested: auto | Actual: brute
Auto Benchmark Completed
Requested: brute | Actual: brute
Brute Benchmark Completed
Requested: kd_tree | Actual: kd_tree
KD-Tree Benchmark Completed
Requested: ball_tree | Actual: ball_tree
Ball-Tree Benchmark Completed


In [10]:
#######################################
# KNN Algorithm Benchmark Summary
#######################################

algorithm_summary = pd.DataFrame()

algorithm_results = [
    auto_results,
    brute_results,
    kd_tree_results,
    ball_tree_results
]

for result in algorithm_results:

    row = {
        "Algorithm": result["Algorithm"],

        "Feature Dimensions": result["Feature Dimensions"][0],
        "Memory (MB)": result["Memory (MB)"][0],

        "Preprocessing Mean (s)": np.mean(result["Preprocessing Time"]),
        "Preprocessing Std (s)": np.std(result["Preprocessing Time"]),
        "Preprocessing Min (s)": np.min(result["Preprocessing Time"]),
        "Preprocessing Max (s)": np.max(result["Preprocessing Time"]),

        "Fit Mean (s)": np.mean(result["Fit Time"]),
        "Fit Std (s)": np.std(result["Fit Time"]),
        "Fit Min (s)": np.min(result["Fit Time"]),
        "Fit Max (s)": np.max(result["Fit Time"]),

        "Prediction Mean (s)": np.mean(result["Prediction Time"]),
        "Prediction Std (s)": np.std(result["Prediction Time"]),
        "Prediction Min (s)": np.min(result["Prediction Time"]),
        "Prediction Max (s)": np.max(result["Prediction Time"]),

        "Total Mean (s)": np.mean(result["Total Time"]),
        "Total Std (s)": np.std(result["Total Time"]),
        "Total Min (s)": np.min(result["Total Time"]),
        "Total Max (s)": np.max(result["Total Time"])
    }

    algorithm_summary = pd.concat(
        [algorithm_summary, pd.DataFrame([row])],
        ignore_index=True
    )

print("=" * 10, "KNN Algorithm Benchmark Summary", "=" * 10)

display(algorithm_summary)

========== KNN Algorithm Benchmark Summary ==========


,Algorithm,Feature Dimensions,Memory (MB),Preprocessing Mean (s),Preprocessing Std (s),Preprocessing Min (s),Preprocessing Max (s),Fit Mean (s),Fit Std (s),Fit Min (s),Fit Max (s),Prediction Mean (s),Prediction Std (s),Prediction Min (s),Prediction Max (s),Total Mean (s),Total Std (s),Total Min (s),Total Max (s)
0,auto,206,3.970001,0.020884,0.004401,0.015201,0.030084,0.001188,0.000359,0.000697,0.001845,0.001024,0.000255,0.000662,0.001495,0.023510,0.004712,0.017258,0.034120
1,brute,206,3.970001,0.019650,0.004476,0.014805,0.027892,0.000926,0.000240,0.000714,0.001515,0.000852,0.000186,0.000663,0.001539,0.021712,0.004691,0.016880,0.030475
2,kd_tree,206,3.970001,0.018275,0.001893,0.014509,0.024278,0.056920,0.008206,0.046449,0.097503,0.001235,0.000271,0.000979,0.001972,0.076690,0.008936,0.062179,0.119075
3,ball_tree,206,3.970001,0.016843,0.001557,0.014584,0.022393,0.030298,0.002374,0.026303,0.035038,0.001557,0.000282,0.001190,0.002309,0.048941,0.003425,0.042765,0.056267
